# 第15章：Ollama - 本地运行大语言模型

本章介绍 Ollama，这是一个用于在本地轻松运行大语言模型（LLM）的工具。我们将学习如何安装 Ollama、下载模型、与模型交互，以及如何将 Ollama 与 LangChain 集成。

**核心知识点**：
- Ollama 安装和基本使用
- 模型下载和管理
- 通过命令行和 API 与模型交互
- LangChain 与 Ollama 集成（ChatOllama）

## 学习目标与环境准备

**学习目标**：
1. 了解 Ollama 的功能和优势
2. 掌握 Ollama 的安装和基本操作
3. 学会下载和管理不同的 LLM 模型
4. 能够将 Ollama 与 LangChain 集成使用

**环境准备**：
- 实际使用需要从 https://ollama.com 下载并安装 Ollama
- 本章通过模拟代码展示 Ollama 的使用方式

## 15.1 Ollama 简介

Ollama 是一个开源工具，让你可以在本地计算机上轻松运行各种大语言模型。它支持 macOS、Linux 和 Windows，提供简单的命令行界面和 REST API。

In [ ]:
print("=== Ollama 主要特点 ===")
print("1. 一键运行 - 使用简单的命令即可启动模型")
print("2. 本地运行 - 数据在本地处理，保护隐私")
print("3. 丰富模型库 - 支持 Llama 3、Mistral、Gemma 等多种模型")
print("4. REST API - 提供简单的 API 接口供程序调用")
print("5. 多平台支持 - 支持 Windows、macOS、Linux")

## 15.2 模拟 Ollama 客户端

让我们创建一个模拟的 Ollama 客户端，来演示 Ollama 的主要功能。

In [ ]:
from typing import Dict, List, Optional, Generator
import time
import json


class MockOllama:
    def __init__(self):
        self.available_models = {
            "llama3": {
                "name": "llama3",
                "size": "4.7 GB",
                "description": "Meta Llama 3，强大的通用模型"
            },
            "llama3:70b": {
                "name": "llama3:70b",
                "size": "40 GB",
                "description": "Llama 3 70B 大模型版本"
            },
            "mistral": {
                "name": "mistral",
                "size": "4.1 GB",
                "description": "Mistral AI 开发的高效模型"
            },
            "gemma:2b": {
                "name": "gemma:2b",
                "size": "1.4 GB",
                "description": "Google Gemma 轻量级模型"
            },
            "qwen2": {
                "name": "qwen2",
                "size": "4.3 GB",
                "description": "通义千问 2，中文优化模型"
            }
        }
        self.installed_models = {}
        self.conversation_history: List[Dict] = []

    def list_models(self) -> List[Dict]:
        print("\n可用的模型：")
        print("-" * 60)
        for name, info in self.available_models.items():
            status = "✓ 已安装" if name in self.installed_models else "  未安装"
            print(f"{name:<15} {info['size']:<10} {status}")
            print(f"  {info['description']}")
        return list(self.available_models.values())

    def pull(self, model_name: str) -> bool:
        if model_name not in self.available_models:
            print(f"错误：模型 '{model_name}' 不存在")
            return False
        
        if model_name in self.installed_models:
            print(f"模型 '{model_name}' 已经安装")
            return True
        
        print(f"正在拉取模型 '{model_name}'...")
        model_info = self.available_models[model_name]
        
        print("下载进度：", end="", flush=True)
        for i in range(1, 11):
            time.sleep(0.2)
            print(f"{i * 10}%", end=" " if i < 10 else "\n", flush=True)
        
        self.installed_models[model_name] = model_info
        print(f"✓ 成功安装模型 '{model_name}'")
        return True

    def _generate_response(self, model: str, prompt: str) -> str:
        if "你好" in prompt or "hello" in prompt.lower():
            return f"你好！我是 {model} 模型。很高兴见到你！有什么我可以帮助你的吗？"
        elif "解释" in prompt or "什么是" in prompt:
            return f"好的，让我来解释一下。基于我的知识库，这是一个重要的概念。简单来说，这个问题涉及到多个方面，需要综合考虑各种因素才能给出完整的答案。"
        elif "代码" in prompt or "写" in prompt and "程序" in prompt:
            return f"当然！这是一个简单的示例代码：\n\ndef hello_world():\n    print('Hello, World!')\n\nhello_world()\n\n这段代码展示了基本的编程结构。"
        elif "翻译" in prompt:
            return f"翻译结果：这是一个模拟的翻译响应。在实际使用中，Ollama 会根据你的输入提供准确的翻译。"
        else:
            return f"我理解你的问题了。作为 {model} 模型，我会尽力提供有帮助的回答。这是一个很好的问题，让我来思考一下..."

    def generate(self, model: str, prompt: str, stream: bool = False) -> str:
        if model not in self.installed_models:
            print(f"错误：模型 '{model}' 未安装。请先运行 pull 命令。")
            return ""
        
        print(f"\n正在使用 {model} 生成响应...")
        response = self._generate_response(model, prompt)
        
        if stream:
            print("流式输出：")
            for char in response:
                print(char, end="", flush=True)
                time.sleep(0.03)
            print()
        else:
            print("响应：")
            print(response)
        
        return response

    def chat(self, model: str, messages: List[Dict]) -> Dict:
        if model not in self.installed_models:
            return {"error": f"模型 '{model}' 未安装"}
        
        last_message = messages[-1]["content"] if messages else ""
        response = self._generate_response(model, last_message)
        
        return {
            "model": model,
            "message": {
                "role": "assistant",
                "content": response
            },
            "done": True
        }

    def delete(self, model_name: str) -> bool:
        if model_name not in self.installed_models:
            print(f"错误：模型 '{model_name}' 未安装")
            return False
        
        del self.installed_models[model_name]
        print(f"✓ 已删除模型 '{model_name}'")
        return True


ollama = MockOllama()
print("模拟 Ollama 客户端已创建！")

## 15.3 查看和下载模型

首先，让我们查看可用的模型并下载一些常用模型。

In [ ]:
# 查看所有可用模型
print("=== 查看可用模型 ===")
ollama.list_models()

# 下载几个常用模型
print("\n=== 下载模型 ===")
ollama.pull("llama3")
ollama.pull("qwen2")
ollama.pull("gemma:2b")

# 再次查看已安装的模型
print("\n=== 当前已安装模型 ===")
print("已安装的模型：")
for name in ollama.installed_models:
    print(f"  ✓ {name}")

## 15.4 使用模型生成文本

现在让我们使用已下载的模型来生成响应。

In [ ]:
# 使用 llama3 模型
print("=== 使用 Llama 3 模型 ===")
response1 = ollama.generate("llama3", "你好，请介绍一下你自己")

# 使用 qwen2 模型（中文优化）
print("\n=== 使用 Qwen 2 模型 ===")
response2 = ollama.generate("qwen2", "请解释什么是人工智能？")

# 流式输出演示
print("\n=== 流式输出演示 ===")
response3 = ollama.generate("gemma:2b", "写一段关于学习的话", stream=True)

## 15.5 多轮对话

Ollama 也支持多轮对话，让我们来模拟一下。

In [ ]:
print("=== 多轮对话演示 ===")

# 初始化对话历史
conversation = [
    {"role": "system", "content": "你是一个乐于助人的AI助手。"}
]

# 第一轮对话
user_msg1 = {"role": "user", "content": "你好，我想学习编程"}
conversation.append(user_msg1)
print(f"\n用户：{user_msg1['content']}")

response1 = ollama.chat("llama3", conversation)
conversation.append(response1["message"])
print(f"助手：{response1['message']['content']}")

# 第二轮对话
user_msg2 = {"role": "user", "content": "应该从什么语言开始？"}
conversation.append(user_msg2)
print(f"\n用户：{user_msg2['content']}")

response2 = ollama.chat("llama3", conversation)
conversation.append(response2["message"])
print(f"助手：{response2['message']['content']}")

## 15.6 模拟 Ollama REST API

Ollama 提供了 REST API，让程序可以方便地调用。让我们模拟这个 API。

In [ ]:
class MockOllamaAPI:
    def __init__(self, ollama_client: MockOllama):
        self.client = ollama_client
        self.base_url = "http://localhost:11434"
    
    def post(self, endpoint: str, data: Dict) -> Dict:
        print(f"\n[API] POST {self.base_url}{endpoint}")
        print(f"[API] 请求数据：{json.dumps(data, ensure_ascii=False)[:100]}...")
        
        if endpoint == "/api/generate":
            response = self.client.generate(data["model"], data["prompt"])
            return {
                "model": data["model"],
                "response": response,
                "done": True
            }
        elif endpoint == "/api/chat":
            return self.client.chat(data["model"], data["messages"])
        elif endpoint == "/api/tags":
            return {
                "models": [
                    {"name": name, "model": name}
                    for name in self.client.installed_models
                ]
            }
        else:
            return {"error": "未知端点"}
    
    def list_models(self):
        return self.post("/api/tags", {})


print("=== Ollama REST API 模拟 ===")
api = MockOllamaAPI(ollama)

# 列出模型
print("\n1. 列出已安装模型（/api/tags）")
models_response = api.list_models()
print(f"响应：{models_response}")

# 生成文本
print("\n2. 生成文本（/api/generate）")
generate_response = api.post("/api/generate", {
    "model": "llama3",
    "prompt": "什么是机器学习？"
})
print(f"响应：{generate_response}")

# 对话
print("\n3. 对话（/api/chat）")
chat_response = api.post("/api/chat", {
    "model": "qwen2",
    "messages": [
        {"role": "user", "content": "用Python写一个快速排序"}
    ]
})
print(f"响应：{chat_response}")

## 15.7 LangChain 集成 - ChatOllama

现在让我们模拟 LangChain 与 Ollama 的集成，使用 ChatOllama 类。

In [ ]:
from typing import List, Optional

class MockChatMessage:
    def __init__(self, content: str, role: str = "user"):
        self.content = content
        self.role = role
    
    def __repr__(self):
        return f"{self.role}: {self.content}"

class MockHumanMessage(MockChatMessage):
    def __init__(self, content: str):
        super().__init__(content, "human")

class MockAIMessage(MockChatMessage):
    def __init__(self, content: str):
        super().__init__(content, "ai")

class MockSystemMessage(MockChatMessage):
    def __init__(self, content: str):
        super().__init__(content, "system")


class MockChatOllama:
    def __init__(self, model: str, ollama_client: MockOllama):
        self.model = model
        self.client = ollama_client
    
    def invoke(self, messages: List[MockChatMessage]) -> MockAIMessage:
        formatted_messages = [
            {"role": msg.role, "content": msg.content}
            for msg in messages
        ]
        response = self.client.chat(self.model, formatted_messages)
        return MockAIMessage(response["message"]["content"])
    
    def stream(self, messages: List[MockChatMessage]):
        response = self.invoke(messages)
        for char in response.content:
            yield char


print("=== LangChain ChatOllama 集成模拟 ===")

# 创建 ChatOllama 实例
chat = MockChatOllama("llama3", ollama)

# 准备消息
messages = [
    MockSystemMessage("你是一个专业的Python编程导师。"),
    MockHumanMessage("如何学习Python？请给出具体建议。")
]

# 调用模型
print("\n--- 单次调用（invoke）---")
response = chat.invoke(messages)
print(f"AI: {response.content}")

# 流式调用
print("\n--- 流式调用（stream）---")
print("AI: ", end="", flush=True)
for chunk in chat.stream([MockHumanMessage("介绍一下Python的特点")]):
    print(chunk, end="", flush=True)
print()

## 15.8 创建简单的聊天应用

让我们结合上面的功能，创建一个简单的命令行聊天应用。

In [ ]:
class SimpleChatApp:
    def __init__(self, ollama_client: MockOllama, model: str = "llama3"):
        self.client = ollama_client
        self.model = model
        self.history: List[Dict] = [
            {"role": "system", "content": "你是一个友好的AI助手。"}
        ]
    
    def send_message(self, user_input: str) -> str:
        self.history.append({"role": "user", "content": user_input})
        response = self.client.chat(self.model, self.history)
        assistant_message = response["message"]["content"]
        self.history.append({"role": "assistant", "content": assistant_message})
        return assistant_message
    
    def clear_history(self):
        self.history = [
            {"role": "system", "content": "你是一个友好的AI助手。"}
        ]
        print("对话历史已清空。")
    
    def print_history(self):
        print("\n" + "="*60)
        print("对话历史：")
        print("="*60)
        for msg in self.history[1:]:  # 跳过 system 消息
            role = "用户" if msg["role"] == "user" else "助手"
            print(f"{role}: {msg['content']}")
        print("="*60 + "\n")


print("=== 简单聊天应用 ===")
app = SimpleChatApp(ollama, "qwen2")

# 模拟对话
print("\n开始对话！（输入 'quit' 退出，'clear' 清空历史）")
test_inputs = [
    "你好！",
    "你能做什么？",
    "告诉我一个笑话",
    "quit"
]

for user_input in test_inputs:
    if user_input.lower() == "quit":
        print("\n用户：quit")
        print("再见！")
        break
    elif user_input.lower() == "clear":
        app.clear_history()
        continue
    
    print(f"\n用户：{user_input}")
    response = app.send_message(user_input)
    print(f"助手：{response}")

# 查看历史
app.print_history()

## 15.9 实际使用指南

以下是在真实环境中使用 Ollama 的实际命令参考。

In [ ]:
print("""
=== Ollama 实际使用命令 ===

# 1. 安装 Ollama
# 访问 https://ollama.com 下载并安装

# 2. 拉取模型
ollama pull llama3
ollama pull qwen2
ollama pull mistral

# 3. 列出已安装模型
ollama list

# 4. 运行模型（交互式）
ollama run llama3

# 5. 直接提问
ollama run llama3 "你好，介绍一下你自己"

# 6. 删除模型
ollama rm llama3

# 7. 复制模型
ollama cp llama3 my-llama3

# 8. 创建自定义模型（使用 Modelfile）
# 编写 Modelfile 后运行：
ollama create my-model -f Modelfile

=== 使用 Ollama API ===

# 生成文本
curl http://localhost:11434/api/generate -d '{"model": "llama3", "prompt": "你好"}'

# 对话
curl http://localhost:11434/api/chat -d '{"model": "llama3", "messages": [{"role": "user", "content": "你好"}]}'

# 列出模型
curl http://localhost:11434/api/tags

=== LangChain 集成 ===

# 安装 LangChain Ollama 集成
pip install langchain-ollama

# 简单使用示例：
# from langchain_ollama import ChatOllama
# from langchain_core.messages import HumanMessage
# 
# chat = ChatOllama(model="llama3")
# response = chat.invoke([HumanMessage(content="你好！")])
# print(response.content)
""")

## 练习

1. **安装和实践**：在你的计算机上安装 Ollama，下载并测试至少 2 个不同的模型。

2. **模型对比**：使用不同的模型（如 Llama 3 和 Qwen 2）回答相同的问题，对比它们的回答风格和质量。

3. **创建 Modelfile**：学习使用 Modelfile 创建自定义模型，尝试修改系统提示词。

4. **构建应用**：使用 LangChain + Ollama 构建一个简单的应用，如文本摘要器或翻译器。